## Scipy and Optimization

In the last notebook we took a tour of the `numpy` and `scipy` stack as it relates to math and statistics. This notebook is going to focus on another subset of `scipy` functionality that is absolutely critical to data analytics. Nearly every statistical algorithm aside from Ordinary Least Squares (OLS) regression cannot be solved algebraically. Because they cannot be solved algebraically, they must be solved **numerically**.

### Numeric Optimization

Let's start with an example of optimization. Let's say that you know the demand function for tickets to watch a local sports franchise. You can write the inverse demand function as 

$$ P = 300 - \frac{1}{2}Q $$

and the total cost function as 

$$ TC = 4000 + 45Q $$

In order to choose the right number of tickets to sell, you need to calculate the quantity of tickets that will maximize profits. We can calculate total revenue as $ TR = P \times Q $, and we can calculate profit as $ \Pi = TR - TC $. This means that our profit function is 

$$ \Pi = 300Q - \frac{1}{2}Q^2 - 4000 - 45Q $$

In order to find the $Q$ associated with the highest achievable level of profit, we can use calculus to find the point at which the rate of change in the profit function is zero ($\frac{\partial\Pi}{\partial Q}=0$).

$$ \frac{\partial\Pi}{\partial Q} = 300 - Q - 45 = 0 \implies Q = 255$$

So we can **algebraically** solve this particular problem. This isn't always the case. Using `scipy`, we can solve this same problem, as well as many algebraically intractable problems that might be more interesting to us.


<img src="https://github.com/dustywhite7/Econ8320/raw/master/SlidesCode/paraboloid.png" width="200" height="200" />


In any optimization problem, we need to find a way to get ourselves to the minimum, and to know when we get there. When we look at the above image, we are able to visually trace the functional shape (looks like a rainbow ice cream cone to me...) and locate the bottom of the function. What we want to do is utilize an algorithm to "trace" our way from an arbitrary starting point within a function to the optimal point in that function. 

In three or fewer dimensions, this is easy. Regressions and statistical models often live in worlds with 100's or 1000's (even millions sometimes) of dimensions. We can't visualize our way to the bottom of those functions! 

The class of algorithm that is used to solve these problems is called **gradient descent**.

<img src="https://github.com/dustywhite7/Econ8320/raw/master/SlidesCode/gradDesc.png" width="400" />

**Gradient descent** is an algorithm that explores the shape of the function, and determines which direction is most likely to lead to the optimal point. Let's focus on minimization. We want to find our way to the *bottom* of a function, and we can use gradient descent to try to get there. Given any starting point, our goal is to check the direction in which we can move downward most quickly, and start moving in that direction. At some distance from our starting point, we will stop and re-evaluate the direction in which we would like to travel. Are we still heading downhill in the steepest direction? If we aren't, then we need to update our behavior.

Our gradient descent algorithm will keep "looking around" and moving down until it reaches a point at which it can no longer move "down" in any meaningful way. That is the stopping point, and is treated as the optimum.


With an intuitive understanding of how optimization will happen computationally, it's time to learn a bit more about the math and the code that will help us to achieve computational optimization.

Consider a function, $f$, with two variables $x$ and $y$. Because there are two input variables in the function, it has two partial derivatives:

$$ \frac{\partial f}{\partial x} \text{ and } \frac{\partial f}{\partial y} $$

Each partial derivative tells us how $f$ changes as we move in a particular dimension **all else equal**. The **gradient**, then, is the vector of all partial derivatives of a given function at any point along the function:

$$ \nabla f = \left[ \begin{matrix} \frac{\partial f}{\partial x} \\ \\ \frac{\partial f}{\partial y} \end{matrix} \right]  $$

We can use the gradient to determine the linear approximation of a function at any given point. Think about the gradient as the mathematical representation of the slope and direction of a hill you are hiking on. If we know the gradient, we know which way is down. As we continue to calculate gradients while walking, we can continue to ensure that we will walk downhill until we reach the bottom of the hill.




The steps of our gradient descent function will be the following:

- Evaluate the gradient of the function
- Find the direction of steepest descent
- Determine how far to move in that direction
- Move to new point
- Reevaluate the gradient
- Stop moving when gradient is within a margin of error from 0

Let's try to implement gradient descent by solving our old profit maximization problem computationally. The very first thing that we need to do is write a Python function that will represent our mathematical function.

In [ ]:
def profit(q):
    p = 300-0.5*q
    tr = p*q
    tc = 4000 + 45*q
    return tr - tc

This function will allow us to calculate profit at any output level based on our assumed total costs and demand curve. With this function, we can quickly calculate the gradient (in this case, just a simple derivative because our function is univariate) by calculating profit at two nearby points, and dividing by the horizontal distance between those points:

In [ ]:
# Gradient at q=200

(profit(201) - profit(199))/2

    55.0



Thus, a one unit increase in output at $Q=200$ results in a $55 increase in profits. This is cool, but it isn't enough for us to find the point of maximum profit (the optimal point). For that, we will need to calculate LOTS of gradients in order to move along the function until we cannot increase profits any further.

Fortunately for us, `scipy` comes with optimization tools that will do all of the heavy lifting of the "search" for the optimal point. All that we need to do is frame our question algorithmically, and let `scipy` do the rest:

In [ ]:
import numpy as np
from scipy.optimize import minimize

We start by importing the `minimize` function from `scipy.optimize`. Hang on! Weren't we working on a MAXIMIZATION problem?? What are we doing here?

Maximization and minimization are the **same thing**. To maximize a function, you can multiply that function by `-1` and then calculate the minimum of the new "upside-down" function. It is functionally equivalent. So, in computational optimization, we always minimize.

### Prepping for optimization

As we prepare to optimize, there are two common problems with our function that we may need to resolve:

1) When using `minimize` we can only pass an array of inputs, so we have to be careful to write our function accordingly
2) Our problem is concave, and so has a maximum
	- We need to restate it as a minimization problem

Problem 1 does not apply, since our function in univariate. In order to make our problem a minimization problem, we can flip our profit maximization function. We will simply return -1 * profit:

In [ ]:
def profit(q):
    p = 300-0.5*q
    tr = p*q
    tc = 4000 + 45*q
    pi =  tr - tc # didn't name it profit because that is our function's name! Don't want to clutter our name space!
    return -1*pi

### Making the call to `minimize`

Now that our function is ready, it is time to minimize! The `minimize` function takes two arguments:
1. Our function that we want to optimize
2. A starting guess (as a vector)

Let's give it a try.

In [2]:
res = minimize(profit, [0]) # provide function and starting inputs
res

NameError: name 'minimize' is not defined

          fun: -28512.499999980355
     hess_inv: array([[1.00000175]])
          jac: array([0.])
      message: 'Optimization terminated successfully.'
         nfev: 21
          nit: 3
         njev: 7
       status: 0
      success: True
            x: array([255.00019821])



That's it! No calculus, no searching, no checking gradients manually. `minimize` simply takes our function and our starting guess and brings us back the optimal choice. We get lots of information stored in the attributes of the `res` object:

- `fun` provides the value of the function (this is -1 times the profit level at the optimal output in our example)
- `hess_inv` and `jac` are measures of gradient and are used to determine how far to go and in which direction
- `message` should be self-explanatory
- `nfev` is the number of times the function (in this case `profit`) was evaluated during the search
- `nit` is the number of iterations it took to find the optimum
- `njev` is the number of times the Jacobian was estimated
- `status` is a code associated with the `message` and `success` atrributes
- `success` tells you whether or not an optimum was found (sometimes it cannot be easily found!)
- `x` probably the most important attribute. This tells us the optimal input value (in our case $Q$), or set of values depending on our function. In a regression context, this could represent the fitted coefficients!

Going forward, you will realize (especially because so many of them print the output as they optimize) just how many libraries in Python use this minimize function behind the scenes. It is used in `statsmodels`, `sklearn`, and many other high-profile libraries! Now that you know where it really happens (in `scipy`!), you'll be better able to troubleshoot the problems that will inevitably arise as you use statistical models.

**Solve-it!**

In this lesson we learned about optimization using SciPy. For the assignment this week, I would like you to build off of your `RegressionModel` class. You will add a Logistic Regression (Logit) method to your class, so that when the `regression_type` parameter is `logit`, Logistic Regression Results are returned. 

Your job is to create the following functionality within your class object:
- a method (call it `logistic_regression`) that estimates the results of logistic regression using your `x` and `y` data frames, and using a likelihood function and gradient descent (DO NOT USE PREBUILT REGRESSION FUNCTIONS).
    - You need to write a function to calculate the Log-likelihood of your model
    - You need to implement gradient descent to find the optimal values of beta
    - You need to use your beta estimates, the model variance, and calculate the standard errors of the coefficients, as well as Z statistics and p-values
    - the results should be stored in a dictionary named `results`, where each variable name (including the intercept if `create_intercept` is `True`) is the key, and the value is another dictionary, with keys for `coefficient`, `standard_error`, `z_stat`, and `p_value`. The coefficient should be the log odds-ratio (which takes the place of the coefficients in OLS)
- a method called `fit_model` that uses the `self.regression_type` attribute to determine whether or not to run an OLS or Logistic Regression using the data provided. This method should call the correct regression method.
- an updated method (call it `summary`) that presents your regression results in a table
    - Columns should be: Variable name, Log odds-ratio value, standard error, z-statistic, and p-value, in that order.
    - Your summary table should have different column titles for OLS and Logistic Regressions! (think if statement...)

You only need to define the class. My code will create an instance of your class (be sure all the names match these instructions, and those from last week!), and provide data to run a regression. I will provide the same data to you, so that you can experiment and make sure that your code is functioning properly.

NOTE: I have created a [primer on Logistic regression](https://github.com/dustywhite7/Econ8320/blob/master/SlidesPDF/9-2%20-%20Logit%20Primer.pdf) to go with this assignment. See the Github slidesPDF folder

Put the code that you would like graded in the cell labeled `#si-exercise`. I recommend copying your code from the last assignment (in chapter 9 about linear regression and `numpy`), and continuing from there. 



In [ ]:
import numpy as np
import pandas as pd
from scipy import stats


class RegressionModel:
    def __init__(self, x, y, create_intercept=True, regression_type="ols",
                 two_sided_p=False):
        if isinstance(x, pd.Series):
            x = x.to_frame()
        if not isinstance(x, pd.DataFrame):
            raise TypeError("x must be a pandas DataFrame.")
        if not isinstance(y, (pd.DataFrame, pd.Series)):
            raise TypeError("y must be a pandas DataFrame or Series.")
        if isinstance(y, pd.DataFrame) and y.shape[1] != 1:
            raise ValueError("y must contain exactly one column.")
        if len(x) != len(y):
            raise ValueError("x and y must have the same number of rows.")
        if not isinstance(create_intercept, (bool, np.bool_)):
            raise TypeError("create_intercept must be True or False.")
        if not isinstance(regression_type, str) or \
                regression_type.strip().lower() not in ("ols", "logit"):
            raise ValueError("regression_type must be 'ols' or 'logit'.")

        self.x = x.copy()
        self.y = y.copy()
        self.create_intercept = bool(create_intercept)
        self.regression_type = regression_type.strip().lower()
        self.two_sided_p = bool(two_sided_p)
        self._results = {}

        if self.create_intercept:
            self.add_intercept()

    def add_intercept(self):
        if self.create_intercept and "intercept" not in self.x.columns:
            self.x["intercept"] = 1
        return self.x

    def _check_data(self):
        if self.x.isnull().any().any() or pd.isnull(np.asarray(self.y)).any():
            raise ValueError("x and y must not contain missing values.")

    # ------------------------------------------------------------------ OLS
    def ols_regression(self):
        self._check_data()
        X = self.x.to_numpy(dtype=float)
        y = np.asarray(self.y, dtype=float).ravel()
        n, k = X.shape

        if n <= k:
            raise ValueError("Need more observations than parameters.")
        if np.linalg.matrix_rank(X) < k:
            raise np.linalg.LinAlgError(
                "x is rank deficient (perfect multicollinearity).")

        xtx_inv = np.linalg.inv(X.T @ X)
        beta = xtx_inv @ X.T @ y
        resid = y - X @ beta
        df_resid = n - k
        sigma2 = (resid @ resid) / df_resid
        se = np.sqrt(np.diag(sigma2 * xtx_inv))
        t_stat = beta / se
        if self.two_sided_p:
            p_val = 2 * stats.t.sf(np.abs(t_stat), df_resid)
        else:
            p_val = stats.t.sf(t_stat, df_resid)

        self._results = {
            name: {"coefficient": float(beta[i]), "standard_error": float(se[i]),
                   "t_stat": float(t_stat[i]), "p_value": float(p_val[i])}
            for i, name in enumerate(self.x.columns)
        }
        return self._results

    # ---------------------------------------------------------------- Logit
    @staticmethod
    def _sigmoid(z):
        return 0.5 * (1 + np.tanh(0.5 * z))

    def log_likelihood(self, beta, X, y):
        """Bernoulli log-likelihood: sum[ y*xb - log(1 + exp(xb)) ]."""
        xb = X @ beta
        return float(np.sum(y * xb - np.logaddexp(0, xb)))

    def _gradient(self, beta, X, y):
        """Gradient of the negative mean log-likelihood."""
        return X.T @ (self._sigmoid(X @ beta) - y) / len(y)

    def logistic_regression(self, max_iter=50000, tol=1e-8):
        self._check_data()
        X = self.x.to_numpy(dtype=float)
        y = np.asarray(self.y, dtype=float).ravel()
        n, k = X.shape

        if n <= k:
            raise ValueError("Need more observations than parameters.")

        # Standardize for stable gradient descent. Only center when an
        # intercept exists; otherwise centering would change the model.
        sd = X.std(axis=0)
        sd[sd == 0] = 1.0
        mu = X.mean(axis=0) if self.create_intercept else np.zeros(k)
        if self.create_intercept:
            i0 = list(self.x.columns).index("intercept")
            mu[i0] = 0.0
        Z = (X - mu) / sd

        # Step size 1/L guarantees descent on this loss
        L = 0.25 * np.sum(Z ** 2) / n
        lr = 1.0 / L

        gamma = np.zeros(k)
        it = 0
        for it in range(max_iter):
            grad = self._gradient(gamma, Z, y)
            if np.linalg.norm(grad) < tol:
                break
            gamma = gamma - lr * grad

        # Back-transform to the original scale
        beta = gamma / sd
        if self.create_intercept:
            others = np.arange(k) != i0
            beta[i0] = gamma[i0] - np.sum(gamma[others] * mu[others] / sd[others])
        self.beta = beta

        # Covariance = inverse Hessian of the negative log-likelihood
        p_hat = self._sigmoid(X @ beta)
        W = p_hat * (1 - p_hat)
        cov = np.linalg.inv(X.T @ (X * W[:, None]))
        se = np.sqrt(np.diag(cov))
        z = beta / se
        p_val = 2 * stats.norm.sf(np.abs(z))

        self._results = {
            name: {"coefficient": float(beta[i]), "standard_error": float(se[i]),
                   "z_stat": float(z[i]), "p_value": float(p_val[i])}
            for i, name in enumerate(self.x.columns)
        }
        return self._results

    # ------------------------------------------------------------ Dispatcher
    def fit_model(self):
        if self.regression_type == "logit":
            return self.logistic_regression()
        return self.ols_regression()

    @property
    def results(self):
        if not self._results:
            self.fit_model()
        return self._results

    # --------------------------------------------------------------- Summary
    def summary(self):
        if self.regression_type == "logit":
            cols = ["Variable name", "coefficient (log odds-ratio)", "standard error",
                    "z-statistic", "p-value"]
            stat_key = "z_stat"
        else:
            cols = ["Variable name", "coefficient value", "standard error",
                    "t-statistic", "p-value"]
            stat_key = "t_stat"

        table = pd.DataFrame(
            [[name, r["coefficient"], r["standard_error"], r[stat_key], r["p_value"]]
             for name, r in self.results.items()],
            columns=cols,
        )
        print(table.to_string(index=False, float_format=lambda v: f"{v:.6f}"))
        return table

In [ ]:
import pandas as pd
import numpy as np

data = pd.read_csv("assignment8Data.csv")

print(data.shape)
print(data.head())

(13712, 95)
   Unnamed: 0     index  year  datanum    serial   hhwt  cpi99  region  \
0           0  20134147  2014        1  730568.0   23.0  0.704      22   
1           1  20134148  2014        1  730568.0   23.0  0.704      22   
2           2  20134149  2014        1  730569.0   43.0  0.704      22   
3           3  20134151  2014        1  730571.0  110.0  0.704      22   
4           4  20134152  2014        1  730571.0  110.0  0.704      22   

   statefip  countyfips  ...  ind5  ind6  ind7  ind8  ind9  ind10  ind11  \
0        31           0  ...     0     0     1     0     0      0      0   
1        31           0  ...     0     0     0     0     0      0      0   
2        31           0  ...     0     0     0     0     0      0      0   
3        31           0  ...     0     0     1     0     0      0      0   
4        31           0  ...     0     0     0     0     0      0      0   

   weekswork  totalhrs     hrwage  
0       52.0    2080.0   9.134615  
1       52.0  

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

data = pd.read_csv("assignment8Data.csv")

x = data.loc[:100, ['sex', 'age', 'educ']]
y = data.loc[:100, 'white']

print("Data loaded")

reg = RegressionModel(
    x, y,
    create_intercept=True,
    regression_type='logit'
)

print("Model created")

result = reg.logistic_regression()

print("Regression completed")
print(pd.DataFrame(result).T)

: 

In [ ]:
import os

print(os.getcwd())

c:\Users\skurz\Desktop\PythonClass\Lesson10


In [ ]:
import sys
import numpy as np
import pandas as pd
import scipy
from scipy import linalg, stats

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)

# Test 1: NumPy matrix multiplication
A = np.array([[1., 2.], [3., 4.]])
B = A @ A
print("\nTest 1 - NumPy:", B)

# Test 2: NumPy linear algebra
print("\nTest 2 - Determinant:", np.linalg.det(A))

# Test 3: SciPy linear algebra
print("\nTest 3 - SciPy:", linalg.det(A))

# Test 4: SciPy statistics
print("\nTest 4 - Statistics:", stats.norm.cdf(0))

# Test 5: Pandas and NumPy integration
df = pd.DataFrame(A, columns=["x", "y"])
print("\nTest 5 - Pandas:")
print(df @ df)

print("\nALL TESTS COMPLETED")

: 

In [ ]:

#si-exercise

import numpy as np
import pandas as pd
from scipy import stats


class RegressionModel:

    def __init__(self, x, y, create_intercept=True,
                 regression_type="ols", two_sided_p=False):

        if isinstance(x, pd.Series):
            x = x.to_frame()

        if not isinstance(x, pd.DataFrame):
            raise TypeError("x must be a pandas DataFrame.")

        if not isinstance(y, (pd.DataFrame, pd.Series)):
            raise TypeError("y must be a pandas DataFrame or Series.")

        if isinstance(y, pd.DataFrame) and y.shape[1] != 1:
            raise ValueError("y must contain exactly one column.")

        if len(x) != len(y):
            raise ValueError("x and y must have the same number of rows.")

        if not isinstance(create_intercept, (bool, np.bool_)):
            raise TypeError("create_intercept must be True or False.")

        if not isinstance(regression_type, str) or \
                regression_type.strip().lower() not in ("ols", "logit"):
            raise ValueError("regression_type must be 'ols' or 'logit'.")

        self.x = x.copy()
        self.y = y.copy()

        self.create_intercept = bool(create_intercept)
        self.regression_type = regression_type.strip().lower()
        self.two_sided_p = bool(two_sided_p)

        self._results = {}

        if self.create_intercept:
            self.add_intercept()

    def add_intercept(self):
        if self.create_intercept and "intercept" not in self.x.columns:
            self.x["intercept"] = 1.0
        return self.x

    def _check_data(self):

        if self.x.isnull().any().any():
            raise ValueError("x contains missing values.")

        if pd.isnull(np.asarray(self.y)).any():
            raise ValueError("y contains missing values.")

        X = self.x.to_numpy(dtype=float)
        y = np.asarray(self.y, dtype=float).ravel()

        if not np.isfinite(X).all() or not np.isfinite(y).all():
            raise ValueError("Data contains invalid values.")

        return X, y

    @staticmethod
    def _sigmoid(z):
        z = np.asarray(z, dtype=float)
        p = np.empty_like(z)

        positive = z >= 0
        p[positive] = 1 / (1 + np.exp(-z[positive]))

        ez = np.exp(z[~positive])
        p[~positive] = ez / (1 + ez)

        return p

    # ------------------------------------------------------------
    # One-sided / two-sided p-value helper
    # ------------------------------------------------------------

    def _calculate_p_values(self, statistic, distribution, df=None):

        statistic = np.asarray(statistic)

        if distribution == "t":
            if self.two_sided_p:
                return 2 * stats.t.sf(np.abs(statistic), df)
            return stats.t.sf(np.abs(statistic), df)

        if distribution == "normal":
            if self.two_sided_p:
                return 2 * stats.norm.sf(np.abs(statistic))
            return stats.norm.sf(np.abs(statistic))

    # ------------------------------------------------------------
    # OLS
    # ------------------------------------------------------------

    def ols_regression(self):

        X, y = self._check_data()

        n, k = X.shape

        if n <= k:
            raise ValueError("Need more observations than parameters.")

        beta, _, rank, _ = np.linalg.lstsq(X, y, rcond=None)

        if rank < k:
            raise np.linalg.LinAlgError("X is rank deficient.")

        resid = y - X @ beta

        df_resid = n - k

        sigma2 = (resid @ resid) / df_resid

        _, s, Vt = np.linalg.svd(X, full_matrices=False)

        covariance = (Vt.T * (1 / s**2)) @ Vt
        covariance *= sigma2

        se = np.sqrt(np.diag(covariance))

        t_stat = beta / se

        p_val = self._calculate_p_values(
            t_stat, "t", df_resid
        )

        self._results = {
            name: {
                "coefficient": float(beta[i]),
                "standard_error": float(se[i]),
                "t_stat": float(t_stat[i]),
                "p_value": float(p_val[i])
            }
            for i, name in enumerate(self.x.columns)
        }

        return self._results

    # ------------------------------------------------------------
    # Logistic regression
    # ------------------------------------------------------------

    def logistic_regression(self, max_iter=100, tol=1e-10):

        X, y = self._check_data()

        n, k = X.shape

        if n <= k:
            raise ValueError("Need more observations than parameters.")

        if not np.all(np.isin(y, [0, 1])):
            raise ValueError("Logistic regression requires binary y.")

        if np.linalg.matrix_rank(X) < k:
            raise np.linalg.LinAlgError("X is rank deficient.")

        beta = np.zeros(k)

        converged = False

        for iteration in range(max_iter):

            p = self._sigmoid(X @ beta)

            W = p * (1 - p)

            gradient = X.T @ (y - p)

            H = X.T @ (X * W[:, None])

            step = np.linalg.solve(H, gradient)

            beta_new = beta + step

            if np.max(np.abs(step)) < tol:
                beta = beta_new
                converged = True
                break

            beta = beta_new

        if not converged:
            raise RuntimeError("Logistic regression did not converge.")

        p = self._sigmoid(X @ beta)

        W = p * (1 - p)

        H = X.T @ (X * W[:, None])

        cov = np.linalg.solve(H, np.eye(k))

        se = np.sqrt(np.diag(cov))

        z_stat = beta / se

        p_val = self._calculate_p_values(
            z_stat, "normal"
        )

        self.beta = beta

        self._results = {
            name: {
                "coefficient": float(beta[i]),
                "standard_error": float(se[i]),
                "z_stat": float(z_stat[i]),
                "p_value": float(p_val[i])
            }
            for i, name in enumerate(self.x.columns)
        }

        return self._results

    # ------------------------------------------------------------
    # Dispatcher
    # ------------------------------------------------------------

    def fit_model(self):

        if self.regression_type == "logit":
            return self.logistic_regression()

        return self.ols_regression()

    # ------------------------------------------------------------
    # Results
    # ------------------------------------------------------------

    @property
    def results(self):

        if not self._results:
            self.fit_model()

        return self._results

    # ------------------------------------------------------------
    # Summary
    # ------------------------------------------------------------

    def summary(self):

        if self.regression_type == "logit":

            cols = [
                "Variable name",
                "coefficient (log odds-ratio)",
                "standard error",
                "z-statistic",
                "p-value"
            ]

            stat_key = "z_stat"

        else:

            cols = [
                "Variable name",
                "coefficient value",
                "standard error",
                "t-statistic",
                "p-value"
            ]

            stat_key = "t_stat"

        table = pd.DataFrame(
            [
                [
                    name,
                    r["coefficient"],
                    r["standard_error"],
                    r[stat_key],
                    r["p_value"]
                ]
                for name, r in self.results.items()
            ],
            columns=cols
        )

        print(table.to_string(
            index=False,
            float_format=lambda v: f"{v:.6f}"
        ))

        return table


In [1]:

import sys
import numpy as np
import pandas as pd
import scipy

print("Python:", sys.executable)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)

data = pd.read_csv("assignment8Data.csv")

x = data.loc[:100, ["sex", "age", "educ"]].to_numpy(dtype=np.float64)
y = data.loc[:100, "white"].to_numpy(dtype=np.float64)

X = np.column_stack((x, np.ones(len(x))))

print("X shape:", X.shape)
print("Y shape:", y.shape)
print("Data loaded successfully")

Python: c:\Users\skurz\miniconda3\envs\ds314\python.exe
NumPy: 2.5.4
Pandas: 3.1.0
SciPy: 1.18.1
X shape: (101, 4)
Y shape: (101,)
Data loaded successfully


In [3]:
import numpy as np
import pandas as pd

data = pd.read_csv("assignment8Data.csv")

x = data.loc[:100, ["sex", "age", "educ"]].to_numpy(dtype=np.float64)
y = data.loc[:100, "white"].to_numpy(dtype=np.float64)

X = np.column_stack((x, np.ones(len(x))))

p = np.full(len(y), 0.5)
W = p * (1 - p)

A = X * W[:, None]

print("Element-wise operation passed")
print("A shape:", A.shape)

Element-wise operation passed
A shape: (101, 4)


In [5]:
np.show_config()


Build Dependencies:
  blas:
    detection method: pkgconfig
    found: true
    include directory: C:/Users/skurz/miniconda3/envs/ds314/Library/include
    lib directory: C:/Users/skurz/miniconda3/envs/ds314/Library/lib
    name: blas
    openblas configuration: unknown
    pc file directory: C:/Users/skurz/miniconda3/envs/ds314/Library/lib/pkgconfig
    version: 3.9.0
  lapack:
    detection method: pkgconfig
    found: true
    include directory: C:/Users/skurz/miniconda3/envs/ds314/Library/include
    lib directory: C:/Users/skurz/miniconda3/envs/ds314/Library/lib
    name: lapack
    openblas configuration: unknown
    pc file directory: C:/Users/skurz/miniconda3/envs/ds314/Library/lib/pkgconfig
    version: 3.9.0
Compilers:
  c:
    commands: cl.exe
    linker: link
    name: msvc
    version: 19.44.35229
  c++:
    commands: cl.exe
    linker: link
    name: msvc
    version: 19.44.35229
  cython:
    commands: cython
    linker: cython
    name: cython
    version: 3.3.0
Machine

In [6]:
np.show_runtime()

OSError: [WinError -1066598273] Windows Error 0xc06d007f